In [ ]:
#| export
# Lazy type-hint evaluation so `Path` doesn't need to be imported
# just to put it in a function signature.
from __future__ import annotations

In [ ]:
#| default_exp daily_summary


# Step 10: Daily digest

![Step 10 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/10-step.png)

**Goal:** once a day, post a summary to Slack — *'today we saw 7 birds, 4 unique species, top visitor was the American Robin.'*

Like every consumer of sightings, the digest talks to the `SightingStorage` contract from lesson 6 — never to SQLite directly.

Four functions live in `bird_watcher/daily_summary.py`:

- `build_daily_summary` — build the Slack Block Kit payload from any storage
- `send_daily_summary` — build and send (or dry-run)
- `schedule_daily_summary` — blocking scheduler that posts once a day (for learning)
- `main` — send one digest and exit, the shape a *real* scheduler (cron) wants

## Step 10.0 — Setup

Same setup as lesson 7: the Slack webhook is a secret, so it comes from `.env`. Empty webhook = dry-run mode.

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os

assert load_dotenv(), "Failed to load .env file"

SLACK_WEBHOOK = os.environ.get("SLACK_WEBHOOK", "")

DATA_FOLDER = Path("../data")
DB_PATH = DATA_FOLDER / "birds.db"

print(f"Slack webhook set: {bool(SLACK_WEBHOOK)}")

## Step 10.1 — Query the last 24 hours

The storage contract gives us `get_sightings` — newest first. It has no "since" filter, and we don't need one: `timestamp` is a real `datetime` (lesson 6 made sure of that), so a list comprehension can compare times directly. Fetch plenty, keep what's recent.

In [ ]:
from datetime import datetime, timedelta

from bird_watcher.persistence import SqliteSightingStorage

storage = SqliteSightingStorage(db_path=DB_PATH, cooldown_minutes=3)

cutoff = datetime.now() - timedelta(hours=24)
recent = [s for s in storage.get_sightings(limit=500) if s.timestamp >= cutoff]

print(f"{len(recent)} sighting(s) in the last 24h")

## Step 10.2 — Group by species, find the top visitor

In [ ]:
from collections import Counter

counts = Counter(s.species for s in recent)
print(f"{len(counts)} unique species")
for species, count in counts.most_common(5):
    print(f"  {species}: {count}")

## Step 10.3 — Wrap as `build_daily_summary`

In [ ]:
#| export
from collections import Counter
from datetime import datetime, timedelta

from bird_watcher.persistence import SightingStorage


def build_daily_summary(
    storage: SightingStorage,
    window_hours: int = 24,
) -> dict:
    """Build a Slack message summarizing the last `window_hours` of sightings.

    Args:
        storage: any SightingStorage — the same contract the web app uses.
        window_hours: how far back to look. Default 24 (one day).

    Returns:
        A Slack Block Kit payload.
    """
    cutoff = datetime.now() - timedelta(hours=window_hours)
    sightings = [s for s in storage.get_sightings(limit=500) if s.timestamp >= cutoff]

    total = len(sightings)
    counts = Counter(s.species for s in sightings)
    unique_species = len(counts)
    top_visitor = counts.most_common(1)[0] if counts else None

    title = f":bird: Daily summary — {total} sighting(s), {unique_species} species"
    if top_visitor:
        title += f", top: *{top_visitor[0]}*"

    blocks: list[dict] = [
        {"type": "header", "text": {"type": "plain_text", "text": title}},
        {
            "type": "section",
            "fields": [
                {"type": "mrkdwn", "text": f"*Window*\nlast {window_hours}h"},
                {"type": "mrkdwn", "text": f"*Total sightings*\n{total}"},
                {"type": "mrkdwn", "text": f"*Unique species*\n{unique_species}"},
            ],
        },
    ]
    if counts:
        species_lines = [
            f"• {species} — {count}" for species, count in counts.most_common(10)
        ]
        blocks.append(
            {
                "type": "section",
                "text": {"type": "mrkdwn", "text": "*Top species*\n" + "\n".join(species_lines)},
            }
        )

    return {"blocks": blocks}


## Step 10.4 — Add `send_daily_summary`

In [ ]:
#| export
from bird_watcher.send_alert import send_alert_to_slack


def send_daily_summary(
    storage: SightingStorage,
    window_hours: int = 24,
    webhook_url: str | None = None,
    debug_prn: bool = False,
) -> bool:
    """Build and send (or dry-run) today's daily summary.

    Args:
        storage: any SightingStorage.
        window_hours: how far back to look.
        webhook_url: optional Slack webhook URL. If omitted, this becomes a dry run.
        debug_prn: if True, print a status line.

    Returns:
        True if sent (or dry-run previewed), False on error.
    """
    payload = build_daily_summary(storage, window_hours=window_hours)
    return send_alert_to_slack(payload, webhook_url=webhook_url, debug_prn=debug_prn)


## Step 10.5 — And the scheduler `schedule_daily_summary`

In [ ]:
#| export
import time

import schedule


def schedule_daily_summary(
    storage: SightingStorage,
    run_at_hour: int = 21,
    webhook_url: str | None = None,
    debug_prn: bool = False,
) -> None:
    """Run a blocking scheduler that posts the daily summary once a day.

    For learning purposes only — production code would use a real cron job
    calling `main` below.

    Args:
        storage: any SightingStorage.
        run_at_hour: hour of day to send (24h format). Default 21 (9pm).
        webhook_url: optional Slack webhook URL. If omitted, every send is a dry run.
        debug_prn: if True, print the next scheduled run.
    """
    job = lambda: send_daily_summary(storage, webhook_url=webhook_url, debug_prn=debug_prn)
    schedule.every().day.at(f"{run_at_hour:02d}:00").do(job)
    if debug_prn:
        print(f"Scheduled daily summary at {run_at_hour:02d}:00 every day")
        print("Press Stop / Ctrl+C to exit")
    while True:
        schedule.run_pending()
        time.sleep(60)


## Step 10.6 — Deploy it: one digest per run

`schedule_daily_summary` keeps a Python process alive all day just to fire once — fine for learning, wasteful for real. Grown-up schedulers flip the shape: **the program sends one digest and exits**, and the *operating system* wakes it up on time. On Linux that's `cron` (or a systemd timer).

So the deployable entrypoint is a `main()` that does exactly one run, configured by environment variables — the same pattern as `bird-watcher-web` in lesson 9. Registered in `pyproject.toml`:

```toml
[project.scripts]
bird-watcher-digest = "bird_watcher.daily_summary:main"
```

And the crontab line that makes it daily at 9pm:

```
0 21 * * *  cd /path/to/bird-watcher && SLACK_WEBHOOK=... uv run bird-watcher-digest
```

In [ ]:
#| export
import os

from bird_watcher.persistence import SqliteSightingStorage


def main():
    """Send one daily summary, then exit — the shape a real scheduler (cron) wants.

    Settings come from environment variables, same as `bird-watcher-web`:

        BIRDS_DB       — path to the SQLite file  (default: data/birds.db)
        SLACK_WEBHOOK  — incoming webhook URL     (empty = dry-run, prints instead)
        WINDOW_HOURS   — how far back to look     (default: 24)
    """
    storage = SqliteSightingStorage(
        db_path=os.environ.get("BIRDS_DB", "data/birds.db"),
        cooldown_minutes=0,  # the digest only reads; cooldown never comes into play
    )

    send_daily_summary(
        storage,
        window_hours=int(os.environ.get("WINDOW_HOURS", "24")),
        webhook_url=os.environ.get("SLACK_WEBHOOK", ""),
        debug_prn=True,
    )

## Acceptance criterion

The digest builds and sends (or dry-runs) against the real database — **and** we can prove the *counting logic* is right with lesson 9's trick: a fake storage with known data gives a known summary. Three sightings, two species, blue jay on top — no database, no Slack.

In [ ]:
from dataclasses import dataclass, field

from bird_watcher.daily_summary import build_daily_summary, send_daily_summary
from bird_watcher.persistence import Sighting, SightingStorage

# --- against the real database ---
payload = build_daily_summary(storage)
assert "blocks" in payload
assert any(b.get("type") == "header" for b in payload["blocks"])

sent = send_daily_summary(storage, webhook_url=SLACK_WEBHOOK, debug_prn=True)
assert sent, "send_daily_summary should return True (dry-run if no webhook)"


# --- against a fake: known data in, known summary out (same trick as lesson 9) ---
@dataclass
class FakeSightingStorage(SightingStorage):
    sightings: list[Sighting] = field(default_factory=list)

    def post_sighting(self, sighting, debug_prn=False):
        sighting.id = len(self.sightings) + 1
        self.sightings.insert(0, sighting)

    def get_sightings(self, species=None, limit=100, offset=0, submitted_by=None):
        rows = [
            s for s in self.sightings
            if (species is None or s.species == species)
            and (submitted_by is None or s.submitted_by == submitted_by)
        ]
        return rows[offset:offset + limit]

    def get_sighting(self, sighting_id):
        return next((s for s in self.sightings if s.id == sighting_id), None)


fake = FakeSightingStorage(cooldown_minutes=0)
for species in ["blue jay", "blue jay", "cardinal"]:
    fake.post_sighting(Sighting(species=species, confidence=0.9, image_bytes=b""))

header = build_daily_summary(fake)["blocks"][0]["text"]["text"]
assert "3 sighting(s)" in header, header
assert "2 species" in header, header
assert "blue jay" in header, header  # the top visitor

print("✅ Daily summary built + sent (or dry-run); fake-storage counts check out")

## 🎉 You finished the bird watcher tutorial!

Ten notebooks. Seven modules. One bird watcher.

- pull photos from a phone camera
- find birds in those photos
- name the species
- log everything to a database
- post alerts to Slack
- serve a web gallery
- send a daily digest

**What's next?** Experiment. Add features. Break stuff. That's how you learn.